In [ ]:
# ============================================================
# CELL 1: Import pipeline from src/
# ============================================================
import sys
import os
sys.path.append('../src')

from url_pipeline import check_url, PHISHTANK_DB
import pandas as pd
import time

print("Phishing URL Detection System — Phishing Short URL Detection System")
print("=" * 50)
print("Pipeline loaded successfully!")
print(f"PhishTank DB: {'Available' if os.path.exists(PHISHTANK_DB) else 'Unavailable'}")

In [ ]:
# ============================================================
# CELL 2: Test 10 URLs — mix of safe, suspicious, phishing
# ============================================================

test_urls = [
    # Known safe sites via short URLs
    "https://tinyurl.com/wikipedia-en",
    "https://bit.ly/3google",
    "https://tinyurl.com/python-docs",

    # Direct safe URLs
    "https://www.google.com",
    "https://www.github.com",

    # Suspicious looking URLs
    "https://cutt.ly/github",
    "https://rb.gy/stackoverflow",

    # Known phishing redirect abuses
    "https://www.google.com/url?q=https://evil-site.com",
    "https://docs.google.com/fake-login-page",
    "https://sites.google.com/view/fake-bank",
]

results_log = []

print("TESTING 10 URLs THROUGH Phishing URL Detection System PIPELINE")
print("=" * 60)

for i, url in enumerate(test_urls, 1):
    print(f"\n[{i}/10] Testing: {url}")
    result = check_url(url, db_path=PHISHTANK_DB, verbose=False)

    results_log.append({
        'No'           : i,
        'Input URL'    : result['input_url'],
        'Expanded URL' : str(result['expanded_url'])[:50] + '...'
                        if result['expanded_url'] and
                        len(str(result['expanded_url'])) > 50
                        else result['expanded_url'],
        'PhishTank'    : result['phishtank_result'],
        'ML Prob'      : f"{result['ml_result']['phishing_prob']}%"
                        if result.get('ml_result') else 'N/A',
        'Verdict'      : result['final_verdict'],
        'Reason'       : result['verdict_reason']
    })

    verdict = result['final_verdict']
    icon    = '' if verdict == 'PHISHING' else (
              ' ' if verdict == 'SUSPICIOUS' else '')
    print(f"  {icon} {verdict} — {result['verdict_reason']}")
    time.sleep(0.5)

print("\n\nFINAL RESULTS TABLE")
print("=" * 60)
display(pd.DataFrame(results_log))

In [ ]:
# ============================================================
# CELL 3: Edge case handling tests
# ============================================================

print("EDGE CASE TESTS")
print("=" * 50)

edge_cases = [
    ("not-a-url-at-all",          "Invalid format"),
    ("",                           "Empty string"),
    ("http://",                    "Incomplete URL"),
    ("https://expired-short.ly",   "Possibly expired"),
]

for url, description in edge_cases:
    result = check_url(url, db_path=PHISHTANK_DB, verbose=False)
    print(f"\nTest  : {description}")
    print(f"Input : '{url}'")
    print(f"Result: {result['final_verdict']}")
    if result.get('error'):
        print(f"Error : {result['error']}")

In [ ]:
# ============================================================
# CELL 4: Full verbose output demo — for presentation
# Shows every step of the pipeline clearly
# ============================================================

demo_url = "https://tinyurl.com/wikipedia-en"
result   = check_url(demo_url, db_path=PHISHTANK_DB, verbose=True)

In [ ]:
# ============================================================
# CELL 5: Summary statistics of all 10 tests
# ============================================================

results_df = pd.DataFrame(results_log)

verdict_counts = results_df['Verdict'].value_counts()

print("RESULTS SUMMARY")
print("=" * 40)
print(f"Total URLs tested : {len(results_df)}")
print()
for verdict, count in verdict_counts.items():
    icon = '' if verdict == 'PHISHING' else (
           ' ' if verdict == 'SUSPICIOUS' else '')
    print(f"  {icon} {verdict:<12}: {count}")

print()
print(f"Detection rate    : "
      f"{(verdict_counts.get('PHISHING', 0) / len(results_df) * 100):.0f}%")
print()
print("System is ready for Streamlit UI in Phase 6!")